# Spatial Filtering and Edge Detection

This notebook demonstrates two classical image-processing tasks:

1. **Noise reduction in the spatial domain** using mean and median filters.
2. **Edge detection** using Sobel and Laplacian operators.

The notebook uses sample grayscale images from `scikit-image`, so it can run without hard-coded Google Colab file paths. You can replace the sample images with your own images when needed.


## 1. Imports and reproducibility

We use OpenCV for filtering and edge operators, NumPy for numerical operations, Matplotlib for visualization, and scikit-image for sample images and image-quality metrics.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage import data
from skimage.metrics import mean_squared_error, peak_signal_noise_ratio, structural_similarity

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("OpenCV:", cv2.__version__)


## 2. Spatial filtering: Gaussian noise reduction

Gaussian noise is generated in floating-point format before clipping to the valid grayscale range. This avoids the unsigned-integer wraparound problem that can occur when negative noise values are converted directly to `uint8`.


In [ ]:
# Load a standard grayscale sample image (uint8, range 0-255)
original_noise_image = data.camera().astype(np.uint8)

def add_gaussian_noise(image, mean=0.0, sigma=25.0, random_generator=None):
    """Add Gaussian noise to a grayscale image and return a uint8 result."""
    if image is None or image.ndim != 2:
        raise ValueError("Expected a valid 2D grayscale image.")
    if sigma < 0:
        raise ValueError("sigma must be non-negative.")

    if random_generator is None:
        random_generator = np.random.default_rng()

    noisy = image.astype(np.float32) + random_generator.normal(
        loc=mean, scale=sigma, size=image.shape
    )
    return np.clip(noisy, 0, 255).astype(np.uint8)

noisy_image = add_gaussian_noise(
    original_noise_image, mean=0, sigma=25, random_generator=rng
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(original_noise_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original image")
axes[1].imshow(noisy_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Image with Gaussian noise")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Mean and median filters

We compare two kernel sizes (`3×3` and `5×5`) for both mean and median filtering. Larger kernels generally remove more noise but may blur fine details.


In [ ]:
filtered_images = {
    "Mean filter (3x3)": cv2.blur(noisy_image, (3, 3)),
    "Mean filter (5x5)": cv2.blur(noisy_image, (5, 5)),
    "Median filter (3x3)": cv2.medianBlur(noisy_image, 3),
    "Median filter (5x5)": cv2.medianBlur(noisy_image, 5),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
display_images = [
    ("Original", original_noise_image),
    ("Gaussian noisy", noisy_image),
    *filtered_images.items(),
]
for ax, (title, image) in zip(axes.flat, display_images):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Quantitative evaluation

We report MSE, PSNR, and SSIM against the original image. Lower MSE and higher PSNR/SSIM generally indicate better restoration. These metrics complement visual inspection and do not replace it.


In [ ]:
def image_quality_metrics(reference, candidate):
    """Compute MSE, PSNR, and SSIM for two same-sized grayscale images."""
    if reference.shape != candidate.shape:
        raise ValueError("Images must have the same dimensions.")

    reference_float = reference.astype(np.float32)
    candidate_float = candidate.astype(np.float32)

    mse = mean_squared_error(reference_float, candidate_float)
    psnr = peak_signal_noise_ratio(reference, candidate, data_range=255)
    ssim = structural_similarity(reference, candidate, data_range=255)

    return {"MSE": mse, "PSNR (dB)": psnr, "SSIM": ssim}

metric_rows = {"Noisy input": image_quality_metrics(original_noise_image, noisy_image)}
for name, result in filtered_images.items():
    metric_rows[name] = image_quality_metrics(original_noise_image, result)

metrics_table = __import__("pandas").DataFrame(metric_rows).T
metrics_table.sort_values("MSE")


## 5. Edge detection: Sobel and Laplacian

For edge detection, we use the standard grayscale `moon` sample image. Sobel estimates first-order intensity gradients in the horizontal and vertical directions. The Laplacian operator uses a second-order derivative to highlight rapid intensity changes.


In [ ]:
edge_image = data.moon().astype(np.uint8)

# Sobel gradients
sobel_x = cv2.Sobel(edge_image, cv2.CV_32F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(edge_image, cv2.CV_32F, 0, 1, ksize=3)

# Gradient magnitude combines the horizontal and vertical responses.
sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
sobel_display = cv2.normalize(
    sobel_magnitude, None, 0, 255, cv2.NORM_MINMAX
).astype(np.uint8)

# Laplacian response
laplacian = cv2.Laplacian(edge_image, cv2.CV_32F, ksize=3)
laplacian_display = cv2.normalize(
    np.abs(laplacian), None, 0, 255, cv2.NORM_MINMAX
).astype(np.uint8)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
edge_results = [
    ("Original image", edge_image),
    ("Sobel X", np.abs(sobel_x)),
    ("Sobel Y", np.abs(sobel_y)),
    ("Sobel gradient magnitude", sobel_display),
]
for ax, (title, image) in zip(axes.flat, edge_results):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(laplacian_display, cmap="gray")
plt.title("Laplacian edge response")
plt.axis("off")
plt.show()


## 6. Key observations

- Mean filtering reduces Gaussian noise by averaging local neighborhoods, but can blur edges.
- Median filtering is an order-statistic method and is often more robust to impulse noise than mean filtering; this notebook compares it on Gaussian noise for completeness.
- Increasing the kernel size strengthens smoothing but can remove fine image details.
- Sobel combines horizontal and vertical first-order gradients to highlight edges.
- Laplacian uses second-order intensity changes and can respond to noise, so smoothing may be useful before applying it in noisier settings.

## 7. Limitations and possible improvements

- The experiments use standard sample images and fixed kernel sizes.
- The quantitative metrics are evaluated only for the Gaussian-noise restoration experiment.
- Future work can add impulse-noise experiments, Gaussian pre-smoothing before Laplacian, threshold-based edge maps, and parameter tuning.
